# 📊 Industry-Oriented Sales Forecasting & Revenue Prediction System
**Project 2 | Data Science | Python + Pandas + NumPy + Matplotlib + Seaborn + Scikit-learn**

---
This notebook walks through every stage of the forecasting pipeline interactively.

In [ ]:
# ── Install missing packages (run once) ────────────────────────────────────
# Uncomment the line below if any package is missing:
# !pip install pandas numpy matplotlib seaborn scikit-learn

import warnings; warnings.filterwarnings('ignore')
import os, sys
sys.path.insert(0, os.getcwd())
import matplotlib
matplotlib.use('inline')          # Jupyter inline mode
%matplotlib inline

from sales_forecasting import (
    generate_sales_data, preprocess,
    plot_sales_overview, plot_seasonality, plot_revenue_breakdown,
    plot_trend_decomposition, train_and_evaluate,
    plot_forecasts, plot_model_comparison,
    plot_feature_importance, plot_30day_future_forecast,
    generate_insights_report
)
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
print('✅ Imports OK')

## 1 · Dataset Generation

In [ ]:
raw_df = generate_sales_data(n_years=3)
print('Shape:', raw_df.shape)
raw_df.head(10)

In [ ]:
print(raw_df.dtypes)
print('\nMissing values:\n', raw_df.isnull().sum())
raw_df.describe()

## 2 · Preprocessing & Feature Engineering

In [ ]:
df = preprocess(raw_df)
print('After preprocessing shape:', df.shape)
df[['date','category','sales_amount','lag_7','rolling_7','sin_month']].head(8)

## 3 · Exploratory Data Analysis

In [ ]:
import matplotlib
matplotlib.use('inline')
%matplotlib inline
import matplotlib.pyplot as plt

# Monthly sales trend
monthly = (raw_df.groupby(['year','month','category'])['sales_amount']
                 .sum().reset_index())
monthly['period'] = pd.to_datetime(monthly[['year','month']].assign(day=1))
palette = sns.color_palette('tab10', 5)

fig, ax = plt.subplots(figsize=(14,5))
for i, (cat, grp) in enumerate(monthly.groupby('category')):
    ax.plot(grp['period'], grp['sales_amount']/1000,
            label=cat, color=palette[i], linewidth=2)
    ax.fill_between(grp['period'], grp['sales_amount']/1000, alpha=0.1, color=palette[i])
ax.set_title('Monthly Sales by Category (₹ thousands)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Sales (₹K)')
ax.legend(); ax.grid(alpha=0.3); sns.despine(ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap
num_cols = ['sales_amount','profit','units_sold','month','is_weekend','is_promotion']
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', square=True, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Distribution of daily sales per category
fig, ax = plt.subplots(figsize=(12,5))
for i, (cat, grp) in enumerate(df.groupby('category')):
    sns.kdeplot(grp['sales_amount']/1000, ax=ax, label=cat,
                color=palette[i], linewidth=2)
ax.set_title('Daily Sales Distribution by Category', fontweight='bold')
ax.set_xlabel('Sales (₹K)'); ax.legend(); sns.despine(ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# Weekday vs weekend boxplot
df2 = df.copy()
df2['Day Type'] = df2['is_weekend'].map({0:'Weekday', 1:'Weekend'})
fig, ax = plt.subplots(figsize=(10,5))
sns.boxplot(x='category', y='sales_amount', hue='Day Type', data=df2,
            palette=['#5C85D6','#E07B39'], ax=ax)
ax.set_title('Weekday vs Weekend Sales Distribution', fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('Sales (₹)'); sns.despine(ax=ax)
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

## 4 · Model Training & Evaluation

In [ ]:
results = train_and_evaluate(df)
metrics = pd.read_csv('outputs/model_metrics.csv')
metrics.style.background_gradient(subset=['R²'], cmap='Greens') \
             .background_gradient(subset=['RMSE'], cmap='Reds_r') \
             .format({'MAE':'{:,.0f}','RMSE':'{:,.0f}','R²':'{:.4f}','MAPE%':'{:.2f}%'})

In [ ]:
# Per-model average R² bar
avg_r2 = metrics.groupby('Model')['R²'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8,4))
avg_r2.plot.bar(ax=ax, color=['#4CAF50','#2196F3','#FF9800','#9C27B0'], edgecolor='white')
ax.set_title('Average R² Score by Model (all categories)', fontweight='bold')
ax.set_ylabel('R² Score'); ax.set_xlabel('')
ax.set_ylim(0, 1); ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15); sns.despine(ax=ax)
plt.tight_layout(); plt.show()

## 5 · Forecast Visualisations

In [ ]:
# Actual vs Forecast for one category inline
cat = 'Electronics'
res = results[cat]['Gradient Boosting']
dates = pd.to_datetime(res['dates'])
fig, ax = plt.subplots(figsize=(13,4))
ax.plot(dates, res['y_test']/1000, label='Actual', color='steelblue', linewidth=1.5)
ax.plot(dates, res['preds']/1000, label='Forecast', color='orangered',
        linewidth=1.5, linestyle='--')
ax.set_title(f'{cat} – Actual vs Forecast  |  R²={res["R2"]:.3f}  MAPE={res["MAPE"]:.1f}%',
             fontweight='bold')
ax.set_ylabel('₹K'); ax.legend(); ax.grid(alpha=0.2); sns.despine(ax=ax)
plt.tight_layout(); plt.show()

## 6 · 30-Day Future Forecast

In [ ]:
plot_30day_future_forecast(df, results)

## 7 · Feature Importance

In [ ]:
plot_feature_importance(results)

## 8 · Business Insights Report

In [ ]:
generate_insights_report(raw_df, results)
with open('outputs/business_insights_report.txt', 'r', encoding='utf-8') as f:
    print(f.read())